# EP3: Buffon's needle on the GPU and Amdahl's law

In EP1 I estimated π with Buffon's needle, in C, on one CPU core. In EP3 I took that same program and
moved the needle loop to the GPU with CUDA. Then I measured how the speedup grows with the number of GPU
cores and compared it with Amdahl's law. With the new speed I also answered EP1's question again: how long
would π with 14 decimals take?

In this notebook:

1. the machine and the code,
2. a few single experiments with the engine,
3. the results of my full experiment, run on my laptop,
4. a shorter run on Colab's GPU,
5. my conclusions.

The code is on GitHub: https://github.com/welt00n/aga511/tree/main/EP3

## What I changed after the EP1 feedback

- **Random numbers.** EP1 used `rand()`. On the GPU I now use cuRAND's Philox generator, with one
  independent stream per chunk of needles. On the CPU I use `drand48()` (48 random bits, period 2.8×10¹⁴).
- **π.** EP1 defined π by hand with 14 decimals. I now use `M_PI`, the full double value.
- **The 14-decimal estimate.** Extrapolating a fitted slope over 19 powers of ten amplifies any small error
  in the slope. I now fix the slope at −1/2, which the theory gives, and measure only the constant k.
  The free fit stays, as a check that the error really falls as N^(−1/2).
- **The run time.** I model it as T = a + bN: a fixed start-up time plus a time per needle.

## How the GPU version works

- **The kernel.** Every GPU thread runs EP1's loop on part of the needles. The needles are cut into chunks;
  each thread takes the next free chunk from a shared counter (`atomicAdd`) until none are left.
- **Cores for Amdahl's law.** I use the SM (streaming multiprocessor) as the "core". To run on exactly
  n SMs, I launch enough blocks to fill the whole GPU; each block checks which SM it landed on, and the blocks
  on the other SMs end at once. The remaining threads share all the needles.
- **Same π on any number of SMs.** Each chunk has its own random stream, so a given seed gives exactly the
  same π on 1 SM or on all of them. Every run checks this before it starts.
- **The clock.** My laptop GPU changes its clock with heat and power, so I also count GPU cycles and convert
  them at the measured running clock. That keeps the speedups independent of the clock.
- **Precision.** The test of each needle can run in double (FP64), float (FP32) or half (FP16) precision.
  The count of crossings is always an integer and π is always computed in double.

## The machine

In [ ]:
!nvidia-smi

## The code

In [ ]:
!git clone -q https://github.com/welt00n/aga511.git /content/aga511 2>/dev/null || git -C /content/aga511 pull -q
%cd /content/aga511/EP3

`buffon.cu` is the engine: one experiment per call, on the GPU or, as in EP1, on one CPU core.

In [ ]:
!cat buffon.cu

`buffon.py` runs the experiments, fits the results and draws the graphs.

In [ ]:
!cat buffon.py

## Trying the engine

The first line describes the GPU. The next two throw 10⁸ needles with the same seed, on all SMs and on
one SM: the same π, in very different times. The last one is EP1 on the CPU.

In [ ]:
!nvcc buffon.cu -o buffon -O2 -arch=native
!./buffon -i
!./buffon -n 1e8 -s 42
!./buffon -n 1e8 -s 42 -m 1
!./buffon -n 1e7 -s 42 -c

## Results of my full experiment

I ran it on my laptop (NVIDIA GeForce RTX 5070 Laptop GPU with 36 SMs, Intel Core Ultra 7 255HX):
the GPU and the CPU in FP64 for 30 minutes each, and FP32 and FP16 for 10 minutes each. Each run sweeps N
from 10⁴ upwards, with several seeds per N and, on the GPU, every SM count from 1 to 36.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

def show_run(folder):
    print((folder / "summary.txt").read_text())
    for name in ("error.png", "amdahl.png", "prediction.png"):
        if (folder / name).exists():
            display(Image(str(folder / name), width=1100))

def laptop_run(name):
    return sorted(Path("runs").glob(f"2026-11-08_*-{name}"))[0]

report = sorted(Path("runs").glob("2026-11-08_*-report"))[-1]
print((report / "report.txt").read_text())
display(Image(str(report / "report.png"), width=1100))

On the left, the real error of every run against N: FP64 and FP32 follow 2.37/√N, while FP16 stops at about
4×10⁻⁴. On the right, the time each run would need for each number of correct decimals, predicted from its
measured speed; the green band is the part I actually measured.

### GPU, double precision (FP64)

- **Error.** The real error follows the experimental error and the theory: slope −0.499 ± 0.009,
  k = 2.38 ± 0.10 (theory 2.374). All 9.7×10¹¹ needles of this run together give
  π = 3.1415942 ± 0.0000024, 0.6σ from the true value.
- **Time.** T = 0.15 s + 1.5×10⁻¹⁰ s × N: CUDA's start-up, then the needles.
- **Amdahl's law.** At N = 3.2×10¹⁰, 36 SMs are 34.3 ± 0.2 times faster than one, and the fit gives
  p = 0.99864 ± 0.00015, so the speedup could never pass about 740.
- **p depends on N.** p is about 0 up to N = 10⁶, where the start-up is almost the whole run; it is 0.78 at
  10⁸, 0.97 at 10⁹ and 0.9986 at 3.2×10¹⁰.
- **The prediction works.** Fitted on N ≤ 3×10⁷ only, it predicts the larger runs: measured/predicted is
  0.99 ± 0.06 for the error and 1.026 ± 0.002 for the time (the clock varies by 1.4%).

In [ ]:
show_run(laptop_run("gpu-fp64"))

### CPU, double precision (FP64): EP1

The error falls as N^(−0.500 ± 0.011), with k = 2.36 ± 0.11. One core throws 5.9×10⁷ needles per second,
so 8 decimals would take about 30 years and 14 decimals 3×10¹³ years.

In [ ]:
show_run(laptop_run("cpu-fp64"))

### The other precisions

- **GPU FP32** throws 1.07×10¹¹ needles per second, 16 times FP64: this GPU has one FP64 unit for every 64
  FP32 units. But float keeps only about 7 digits, so it can never give 14 decimals.
- **GPU FP16** is not faster than FP32 (that would need two half numbers per instruction), and its error stops
  at about 3.8×10⁻⁴: rounding biases each needle's test.
- **On the CPU**, FP32 is about as fast as FP64 (1.1×), because the CPU has full double-precision units;
  FP16 is emulated and three times slower.
- FP64 and FP32 use the same random numbers, so their errors are practically identical.

In [ ]:
for name in ("gpu-fp32", "gpu-fp16", "cpu-fp32", "cpu-fp16"):
    show_run(laptop_run(name))

## A run on Colab's GPU

A shorter run (5 minutes) on the GPU Colab provides, to show the code also works here. Colab's T4 has 40 SMs
and runs FP64 at 1/32 of FP32, so its numbers differ from my laptop's.

In [ ]:
!python buffon.py --budget 5m

In [ ]:
colab_run = sorted(Path("runs").glob("*-gpu-fp64"))[-1]
show_run(colab_run)

## Conclusions

1. **The real error behaves as expected.** It follows the experimental error and the theory, 2.37/√N, on
   both devices, and all the GPU's needles together give π within 0.6σ of the true value.
2. **14 decimals are out of reach.** They need (5.6 ± 0.5)×10²⁸ needles: 2.7×10¹¹ years on my GPU, about
   20 times the age of the universe, and 3×10¹³ years on the CPU. Even 8 decimals would take 99 days on the
   GPU and 30 years on the CPU.
3. **The GPU is 112 times faster than EP1 in FP64**, and 1820 times in FP32. Across SMs, 36 SMs are 34.3
   times faster than one: Amdahl's law with p = 0.9986.
4. **p is not a property of the program alone.** The same code has p ≈ 0 for 10⁶ needles and p ≈ 0.999
   for 3×10¹⁰, because the fixed start-up time stops mattering as the work grows.
5. **The run time grows almost exactly as N** (exponent 1.013 ± 0.005 on the GPU, 0.997 ± 0.001 on the CPU),
   which justifies extrapolating it.
6. **Precision is a trade-off.** Lower precision is much faster on this GPU, but only double precision can
   support many decimals; FP16 is limited to about 3.